In [ ]:
# Cell 1 // setup
!pip install -q transformers datasets accelerate peft[safe] sentencepiece sacremoses

from pathlib import Path
import os
import re
import pandas as pd
import numpy as np
import torch
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader

# huggingface / training helpers
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

# mount drive (if using Drive)
from google.colab import drive
drive.mount('/content/drive')

# paths
DRIVE_RAW = Path('/content/drive/My Drive/Gender Neutral/data/raw/')
WORKDIR = Path('/content/data')
OUTPUT_DIR = Path('/content/mt5_lora_output_subtaskA')
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete. DRIVE_RAW:", DRIVE_RAW, "WORKDIR:", WORKDIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 15.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete. DRIVE_RAW: /content/drive/My Drive/Gender Neutral/data/raw WORKDIR: /content/data


In [ ]:
# Cell 2 // load & map datasets (keeps originals, creates mapped views per language)
import os, re
from collections import defaultdict

BASE_PATH = str(DRIVE_RAW) + "/"   # ensure trailing slash for previous code compatibility
DATA_DIRS = {
    "inclusive": os.path.join(BASE_PATH, "Inclusive"),
    "gender_neutral_pairs": os.path.join(BASE_PATH, "Gender Neutral Pairs"),
    "raw": BASE_PATH
}

LANG_MAP = {
    "english": "en", "en": "en",
    "german": "de", "de": "de",
    "spanish": "es", "es": "es",
    "tamil": "ta", "ta": "ta",
    "kannada": "kn", "kn": "kn"
}

def slugify(text):
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", text.lower())).strip("_")

def normalize_cols(df):
    return {c.lower().strip(): c for c in df.columns}

def extract_lang_from_col(col):
    m = re.search(r"\(([^)]+)\)", col.lower())
    if m:
        return LANG_MAP.get(m.group(1).strip(), None)
    for k in LANG_MAP:
        if k in col.lower():
            return LANG_MAP[k]
    return None

def create_mapped_views_for_file(df):
    cols = normalize_cols(df)
    mapped = {}

    # ENGLISH pairs detection
    if "original terms" in cols and "inclusive terms" in cols:
        out = pd.DataFrame({
            "source_text": df[cols["original terms"]],
            "target_text": df[cols["inclusive terms"]],
        })
        out["pair_id"] = range(1, len(out) + 1)
        mapped["en"] = out

    # English sentences detection
    if "non-inclusive" in cols and "inclusive" in cols:
        out = pd.DataFrame({
            "source_text": df[cols["non-inclusive"]],
            "target_text": df[cols["inclusive"]],
        })
        out["pair_id"] = range(1, len(out) + 1)
        mapped.setdefault("en", out)

    # Multi-lingual columns detection
    lang_roles = defaultdict(dict)
    for c in df.columns:
        role = None
        cl = c.lower()
        if "original" in cl or "non-inclusive" in cl or "original terms" in cl:
            role = "source"
        if "inclusive" in cl or "corrected translation" in cl or "inclusive terms" in cl:
            role = "target"
        lang = extract_lang_from_col(c)
        if role and lang:
            lang_roles[lang][role] = c

    # HARD FIX: Tamil "Corrected Translation(Inclusive Terms)" -> map to ta target
    for c in df.columns:
        if "corrected translation" in c.lower():
            lang_roles["ta"]["target"] = c
        # catch "inclusive (Tamil)" variants too
        if "inclusive" in c.lower() and ("tamil" in c.lower() or "(tamil)" in c.lower()):
            lang_roles["ta"]["target"] = c

    for lang, roles in lang_roles.items():
        if "source" in roles and "target" in roles:
            out = pd.DataFrame({
                "source_text": df[roles["source"]],
                "target_text": df[roles["target"]],
            })
            out["pair_id"] = range(1, len(out) + 1)
            mapped[lang] = out

    # cleanup (drop rows where both empty)
    for k in list(mapped.keys()):
        mdf = mapped[k].fillna("").astype(str)
        mdf = mdf[~((mdf["source_text"].str.strip() == "") & (mdf["target_text"].str.strip() == ""))].reset_index(drop=True)
        mapped[k] = mdf[["pair_id", "source_text", "target_text"]]

    return mapped

# load files
datasets = {}
collections = defaultdict(list)

def list_csvs(d):
    return [os.path.join(d,f) for f in os.listdir(d) if f.endswith(".csv")] if os.path.exists(d) else []

for group, path in DATA_DIRS.items():
    for fp in list_csvs(path):
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print("Failed to read", fp, e); continue
        name = os.path.splitext(os.path.basename(fp))[0]
        ftype = "pairs" if any(k in name.lower() for k in ["pair","term","replacement"]) else "sentence"
        key = f"{ftype}_{slugify(name)}"
        mapped = create_mapped_views_for_file(df)
        datasets[key] = {"df": df, "mapped": mapped, "type": ftype, "file": fp}
        for lang in mapped.keys():
            collections[f"{ftype}_{lang}"].append(key)

# summary
print("Loaded dataset keys:", len(datasets))
for k,v in collections.items():
    print(f"  {k}: {len(v)} file(s)")
# keep variables datasets & collections for later cells


KeyboardInterrupt: 

In [ ]:
print("\n=== All mapped datasets previews ===")
for ds_key, entry in datasets.items():
    if entry.get("mapped"):
        for lang_code, mdf in entry["mapped"].items():
            print(f"\nDataset Key: {ds_key} | Mapped Language: {lang_code} | Rows: {len(mdf)}")
            display(mdf.head(3))

In [ ]:
# Cell 3 // create merged CSVs needed by later cells
from pathlib import Path
WORKDIR = Path('/content/data')
WORKDIR.mkdir(parents=True, exist_ok=True)

# Build merged non-de: all mapped views whose lang != 'de'
parts = []
for ds_key, info in datasets.items():
    mapped = info.get("mapped", {})
    for lang_code, mdf in mapped.items():
        if lang_code == "de":
            continue
        tmp = mdf.copy()
        tmp["lang"] = lang_code
        tmp["dataset_key"] = ds_key
        parts.append(tmp)

if parts:
    df_nonde = pd.concat(parts, ignore_index=True)
else:
    df_nonde = pd.DataFrame(columns=["pair_id","source_text","target_text","lang","dataset_key"])

nonde_path = WORKDIR / "subtaskB_merged_nonde.csv"
df_nonde.to_csv(nonde_path, index=False)
print("Wrote non-de merged to:", nonde_path, "rows:", len(df_nonde))

# Build german file if any mapped german data exists
parts_de = []
for ds_key, info in datasets.items():
    mapped = info.get("mapped", {})
    if "de" in mapped:
        mdf = mapped["de"].copy()
        mdf["lang"] = "de"
        mdf["dataset_key"] = ds_key
        parts_de.append(mdf)

de_path = WORKDIR / "subtaskB_de_prepared.csv"
if parts_de:
    df_de = pd.concat(parts_de, ignore_index=True)
    df_de.to_csv(de_path, index=False)
    print("Wrote de prepared to:", de_path, "rows:", len(df_de))
else:
    # ensure empty file not required; we create an empty placeholder to keep downstream code robust
    pd.DataFrame(columns=["pair_id","source_text","target_text","lang","dataset_key"]).to_csv(de_path, index=False)
    print("No German mapped data found; created empty placeholder at:", de_path)


In [ ]:
# Cell 4 // Prepare dataset for Subtask A (sentence rewriting) - build lang-tagged inputs & save
from pathlib import Path
import pandas as pd

WORKDIR = Path('/content/data')
TRAIN_FILE = WORKDIR / 'subtaskB_merged_nonde.csv'   # merged non-de
GERMAN_FILE = WORKDIR / 'subtaskB_de_prepared.csv'   # german prepared (may be empty)
PREPARED_PATH = WORKDIR / 'subtaskA_training_prepared.csv'

# Load merged non-de
df_main = pd.read_csv(TRAIN_FILE)
print("Loaded non-de merged shape:", df_main.shape)

# Optionally append german (if present and you want to include)
INCLUDE_GERMAN_IF_PRESENT = True
if INCLUDE_GERMAN_IF_PRESENT:
    df_de = pd.read_csv(GERMAN_FILE)
    if len(df_de) > 0:
        df_main = pd.concat([df_main, df_de], ignore_index=True, sort=False)
        print("Appended German prepared rows:", len(df_de))

# Filter rows: require source_text present and target_text non-empty
df_main = df_main[df_main['source_text'].notna()].reset_index(drop=True)
df_main['target_text'] = df_main['target_text'].astype(str)
df_main = df_main[df_main['target_text'].str.strip() != ''].reset_index(drop=True)
print("After filtering, rows:", len(df_main))

# Build input_text with language tag for mT5
def lang_tagged_input(row):
    lang = row.get('lang', 'en')
    if pd.isna(lang) or not isinstance(lang, str) or len(lang.strip()) == 0:
        lang = 'en'
    return f"<lang:{lang}> " + str(row['source_text']).strip()

df_main['input_text'] = df_main.apply(lang_tagged_input, axis=1)
df_main['target_text'] = df_main['target_text'].astype(str).str.strip()

# Save prepared training CSV
df_main = df_main[['pair_id','lang','dataset_key','input_text','target_text']]
df_main.to_csv(PREPARED_PATH, index=False)
print("Prepared training CSV saved to:", PREPARED_PATH, "rows:", len(df_main))
display(df_main.sample(min(6, len(df_main))))


In [ ]:
# Cell 5 // HF Dataset and tokenization wrapper
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

PREPARED_PATH = Path('/content/data/subtaskA_training_prepared.csv')
df = pd.read_csv(PREPARED_PATH)
print("Loaded prepared CSV rows:", len(df))

# Convert to HF Dataset
hf_ds = HFDataset.from_pandas(df, preserve_index=False)

# tokenizer (example using mt5-base)
MODEL_NAME = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# tokenization function
max_input_length = 128
max_target_length = 128

def tokenize_fn(ex):
    model_inputs = tokenizer(ex["input_text"], truncation=True, padding="max_length", max_length=max_input_length)
    labels = tokenizer(ex["target_text"], truncation=True, padding="max_length", max_length=max_target_length)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

hf_ds = hf_ds.map(tokenize_fn, batched=True, remove_columns=list(df.columns))
hf_ds = hf_ds.rename_column("input_text", "input_text") if "input_text" in hf_ds.column_names else hf_ds
hf_ds.set_format(type="torch")

print("Tokenized dataset columns:", hf_ds.column_names)
print("Sample tokenized shape:", {c: hf_ds[0][c].shape for c in hf_ds.column_names if c in hf_ds[0]})


In [ ]:
# Cell 6 // model + collator + trainer (FP32, latest HF-compatible)

from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

MODEL_NAME = "google/mt5-base"

print("Loading model (this may take time)...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    tie_word_embeddings=False  # silence shared.weight warnings
)

# Data collator (this now carries tokenizer info)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# =======================
# TRAINING ARGUMENTS
# =======================
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # batching
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,

    # training
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_steps=500,   # ✅ replaces warmup_ratio

    # logging & saving
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # evaluation (new API)
    eval_strategy="no",

    # precision (explicit FP32)
    fp16=False,
    bf16=False,

    # misc
    report_to="none",
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_ds,
    data_collator=data_collator,  # tokenizer inferred here
)

print("Trainer ready (FP32, latest HF API). Call trainer.train() to start.")


In [ ]:
# Cell 7 // quick checks & batch sanity test (FP32-safe)

from torch.utils.data import DataLoader

dataloader = DataLoader(
    hf_ds,
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator
)

batch = next(iter(dataloader))

print("Batch keys:", batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)

# sanity checks
assert batch["input_ids"].dtype == torch.long
assert batch["labels"].dtype == torch.long

print("Sanity check passed. FP32 pipeline is correct.")


# RETRY


In [1]:
# Cell 1
# !pip install -q transformers datasets accelerate peft[safe] sentencepiece sacremoses

!pip uninstall -y torch torchvision torchaudio accelerate peft transformers

# Install compatible versions
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2
!pip install transformers==4.38.2 accelerate==0.28.0 peft==0.10.0 datasets sentencepiece


Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2
Found existing installation: torchaudio 2.2.2
Uninstalling torchaudio-2.2.2:
  Successfully uninstalled torchaudio-2.2.2
Found existing installation: accelerate 0.28.0
Uninstalling accelerate-0.28.0:
  Successfully uninstalled accelerate-0.28.0
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchvision-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.4 kB)
  Using cached nvidia_cuda_nvrtc_cu

In [1]:
!pip uninstall -y torch torchvision torchaudio bitsandbytes triton peft transformers accelerate
!pip install torch==2.2.2
!pip install transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 numpy==1.26.4

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
Found existing installation: accelerate 0.29.3
Uninstalling accelerate-0.29.3:
  Successfully uninstalled accelerate-0.29.3
  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl (755.5 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, which is not installed.
timm 1.0.25 requires torchvision, which is not installed

In [2]:

from pathlib import Path
import os, re
import pandas as pd
# import numpy as np
import torch
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RAW = Path('/content/drive/My Drive/Gender Neutral/data/raw/')
WORKDIR = Path('/content/data')
OUTPUT_DIR = Path('/content/mt5_lora_output_subtaskA')
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Paths -> DRIVE_RAW:", DRIVE_RAW, "WORKDIR:", WORKDIR, "OUTPUT_DIR:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Paths -> DRIVE_RAW: /content/drive/My Drive/Gender Neutral/data/raw WORKDIR: /content/data OUTPUT_DIR: /content/mt5_lora_output_subtaskA


In [3]:
!pip install numpy==1.26.4

In [5]:
import numpy as np
import bitsandbytes

ModuleNotFoundError: No module named 'triton.ops'

In [12]:
# Clean environment for MT0-XL LoRA training
!pip uninstall -y torchvision torchaudio sentence-transformers

!pip install -q \
numpy==1.26.4 \
transformers==4.40.2 \
peft==0.10.0 \
accelerate==0.29.3 \
bitsandbytes==0.43.1 \
triton==2.2.0

Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2
Found existing installation: torchaudio 2.2.2
Uninstalling torchaudio-2.2.2:
  Successfully uninstalled torchaudio-2.2.2
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3


In [4]:
# Cell 2
import os, re
from collections import defaultdict

BASE_PATH = str(DRIVE_RAW) + "/"
DATA_DIRS = {
    "inclusive": os.path.join(BASE_PATH, "Inclusive"),
    "gender_neutral_pairs": os.path.join(BASE_PATH, "Gender Neutral Pairs"),
    "raw": BASE_PATH
}

LANG_MAP = {
    "english": "en", "en": "en",
    "german": "de", "de": "de",
    "spanish": "es", "es": "es",
    "tamil": "ta", "ta": "ta",
    "kannada": "kn", "kn": "kn"
}

def slugify(text):
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", text.lower())).strip("_")

def normalize_cols(df):
    return {c.lower().strip(): c for c in df.columns}

def extract_lang_from_col(col):
    m = re.search(r"\(([^)]+)\)", col.lower())
    if m:
        return LANG_MAP.get(m.group(1).strip(), None)
    for k in LANG_MAP:
        if k in col.lower():
            return LANG_MAP[k]
    return None

def create_mapped_views_for_file(df):
    cols = normalize_cols(df)
    mapped = {}

    if "original terms" in cols and "inclusive terms" in cols:
        out = pd.DataFrame({
            "source_text": df[cols["original terms"]],
            "target_text": df[cols["inclusive terms"]],
        })
        out["pair_id"] = range(1, len(out) + 1)
        mapped["en"] = out

    if "non-inclusive" in cols and "inclusive" in cols:
        out = pd.DataFrame({
            "source_text": df[cols["non-inclusive"]],
            "target_text": df[cols["inclusive"]],
        })
        out["pair_id"] = range(1, len(out) + 1)
        mapped.setdefault("en", out)

    lang_roles = defaultdict(dict)
    for c in df.columns:
        role = None
        cl = c.lower()
        if "original" in cl or "non-inclusive" in cl or "original terms" in cl:
            role = "source"
        if "inclusive" in cl or "corrected translation" in cl or "inclusive terms" in cl:
            role = "target"
        lang = extract_lang_from_col(c)
        if role and lang:
            lang_roles[lang][role] = c

    for c in df.columns:
        if "corrected translation" in c.lower():
            lang_roles["ta"]["target"] = c
        if "inclusive" in c.lower() and ("tamil" in c.lower() or "(tamil)" in c.lower()):
            lang_roles["ta"]["target"] = c

    for lang, roles in lang_roles.items():
        if "source" in roles and "target" in roles:
            out = pd.DataFrame({
                "source_text": df[roles["source"]],
                "target_text": df[roles["target"]],
            })
            out["pair_id"] = range(1, len(out) + 1)
            mapped[lang] = out

    for k in list(mapped.keys()):
        mdf = mapped[k].fillna("").astype(str)
        mdf = mdf[~((mdf["source_text"].str.strip() == "") & (mdf["target_text"].str.strip() == ""))].reset_index(drop=True)
        mapped[k] = mdf[["pair_id", "source_text", "target_text"]]

    return mapped

datasets = {}
collections = defaultdict(list)

def list_csvs(d):
    return [os.path.join(d,f) for f in os.listdir(d) if f.endswith(".csv")] if os.path.exists(d) else []

for group, path in DATA_DIRS.items():
    for fp in list_csvs(path):
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print("Failed to read", fp, e); continue
        name = os.path.splitext(os.path.basename(fp))[0]
        ftype = "pairs" if any(k in name.lower() for k in ["pair","term","replacement"]) else "sentence"
        key = f"{ftype}_{slugify(name)}"
        mapped = create_mapped_views_for_file(df)
        datasets[key] = {"df": df, "mapped": mapped, "type": ftype, "file": fp}
        for lang in mapped.keys():
            collections[f"{ftype}_{lang}"].append(key)

print("Loaded dataset keys:", len(datasets))
for k,v in collections.items():
    print(f"  {k}: {len(v)} file(s)")


Loaded dataset keys: 11
  pairs_en: 6 file(s)
  sentence_en: 2 file(s)
  pairs_es: 1 file(s)
  pairs_ta: 1 file(s)
  pairs_kn: 1 file(s)


In [5]:
# Cell 3
from pathlib import Path
WORKDIR = Path('/content/data')
WORKDIR.mkdir(parents=True, exist_ok=True)

parts = []
for ds_key, info in datasets.items():
    mapped = info.get("mapped", {})
    for lang_code, mdf in mapped.items():
        if lang_code == "de":
            continue
        tmp = mdf.copy()
        tmp["lang"] = lang_code
        tmp["dataset_key"] = ds_key
        tmp = tmp.rename(columns={"source_text":"source_text","target_text":"target_text"})
        parts.append(tmp)

if parts:
    df_nonde = pd.concat(parts, ignore_index=True)
else:
    df_nonde = pd.DataFrame(columns=["pair_id","source_text","target_text","lang","dataset_key"])

nonde_path = WORKDIR / "subtaskB_merged_nonde.csv"
df_nonde.to_csv(nonde_path, index=False)
print("Wrote non-de merged to:", nonde_path, "rows:", len(df_nonde))

parts_de = []
for ds_key, info in datasets.items():
    mapped = info.get("mapped", {})
    if "de" in mapped:
        mdf = mapped["de"].copy()
        mdf["lang"] = "de"
        mdf["dataset_key"] = ds_key
        parts_de.append(mdf)

de_path = WORKDIR / "subtaskB_de_prepared.csv"
if parts_de:
    df_de = pd.concat(parts_de, ignore_index=True)
    df_de.to_csv(de_path, index=False)
    print("Wrote de prepared to:", de_path, "rows:", len(df_de))
else:
    pd.DataFrame(columns=["pair_id","source_text","target_text","lang","dataset_key"]).to_csv(de_path, index=False)
    print("No German mapped data found; empty placeholder created at:", de_path)


Wrote non-de merged to: /content/data/subtaskB_merged_nonde.csv rows: 7381
No German mapped data found; empty placeholder created at: /content/data/subtaskB_de_prepared.csv


In [6]:
# Cell 4
from pathlib import Path
WORKDIR = Path('/content/data')
TRAIN_FILE = WORKDIR / 'subtaskB_merged_nonde.csv'
GERMAN_FILE = WORKDIR / 'subtaskB_de_prepared.csv'
PREPARED_PATH = WORKDIR / 'subtaskA_training_prepared.csv'

df_main = pd.read_csv(TRAIN_FILE)
print("Loaded non-de merged shape:", df_main.shape)

INCLUDE_GERMAN_IF_PRESENT = True
if INCLUDE_GERMAN_IF_PRESENT:
    df_de = pd.read_csv(GERMAN_FILE)
    if len(df_de) > 0:
        df_main = pd.concat([df_main, df_de], ignore_index=True, sort=False)
        print("Appended German prepared rows:", len(df_de))

df_main = df_main[df_main['source_text'].notna()].reset_index(drop=True)
df_main['target_text'] = df_main['target_text'].astype(str)
df_main = df_main[df_main['target_text'].str.strip() != ''].reset_index(drop=True)
print("After filtering, rows:", len(df_main))

def lang_tagged_input(row):
    lang = row.get('lang', 'en')
    if pd.isna(lang) or not isinstance(lang, str) or len(lang.strip()) == 0:
        lang = 'en'
    return f"<lang:{lang}> " + str(row['source_text']).strip()

df_main['input_text'] = df_main.apply(lang_tagged_input, axis=1)
df_main['target_text'] = df_main['target_text'].astype(str).str.strip()

df_main = df_main[['pair_id','lang','dataset_key','input_text','target_text']]
df_main.to_csv(PREPARED_PATH, index=False)
print("Prepared training CSV saved to:", PREPARED_PATH, "rows:", len(df_main))
display(df_main.sample(min(6, len(df_main))))


Loaded non-de merged shape: (7381, 5)
After filtering, rows: 7381
Prepared training CSV saved to: /content/data/subtaskA_training_prepared.csv rows: 7381


,pair_id,lang,dataset_key,input_text,target_text
6796,109,kn,pairs_gender_neutral_pairs_kannada_xlsx_sheet1,<lang:kn> ದಾನಿ ಪುರುಷ,ದಾನಿ
1894,623,en,pairs_gender_inclusive_sentence_pairs_kannada_...,<lang:en> Every congressman should attend the ...,Every member of congress should attend the mee...
6599,605,en,pairs_gender_neutral_pairs_kannada_xlsx_sheet1,<lang:en> stockmanship,livestock raising skill
1737,466,en,pairs_gender_inclusive_sentence_pairs_kannada_...,<lang:en> The maid service was quick and effic...,The house cleaning service was quick and effic...
3200,856,en,sentence_gender_inclusive_data_tamil_xlsx_sheet1,"<lang:en> If you have any questions, please as...","If you have any questions, please ask the supe..."
2777,433,en,sentence_gender_inclusive_data_tamil_xlsx_sheet1,"<lang:en> As an anchorman, he has a deep voice...","As an anchor, they have a voice well-suited fo..."


In [7]:
# # Cell 5
# from datasets import Dataset as HFDataset
# from transformers import AutoTokenizer, DataCollatorForSeq2Seq

# PREPARED_PATH = Path('/content/data/subtaskA_training_prepared.csv')
# df = pd.read_csv(PREPARED_PATH)
# print("Loaded prepared CSV rows:", len(df))

# MODEL_NAME = "google/mt5-base"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# max_input_length = 128
# max_target_length = 128

# hf_ds = HFDataset.from_pandas(df, preserve_index=False)

# def tokenize_fn(examples):
#     model_inputs = tokenizer(examples["input_text"], truncation=True, padding="max_length", max_length=max_input_length)
#     labels = tokenizer(examples["target_text"], truncation=True, padding="max_length", max_length=max_target_length)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

# hf_ds = hf_ds.map(tokenize_fn, batched=True, remove_columns=list(df.columns))
# hf_ds.set_format(type="torch")
# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=None, padding=True)

# print("Tokenized dataset columns:", hf_ds.column_names)
# print("Sample shapes:", {c: hf_ds[0][c].shape for c in hf_ds.column_names if c in hf_ds[0]})

# Cell 5 (REPLACE)
# from datasets import Dataset as HFDataset
# from transformers import AutoTokenizer, DataCollatorForSeq2Seq

# PREPARED_PATH = Path('/content/data/subtaskA_training_prepared.csv')
# df = pd.read_csv(PREPARED_PATH)
# print("Loaded prepared CSV rows:", len(df))

# MODEL_NAME = "google/mt5-base"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# # --- Add explicit language tokens so "<lang:en>" is a known token ---
# # list tokens to add; keep same format you used in inputs ("<lang:en>" etc.)
# LANG_TOKENS = ["<lang:en>", "<lang:de>", "<lang:es>", "<lang:ta>", "<lang:kn>"]

# # Add tokens to tokenizer (idempotent: only actually adds missing tokens)
# added = tokenizer.add_tokens([t for t in LANG_TOKENS if t not in tokenizer.get_vocab()])
# print("New tokens added:", added)

# # Save tokenizer to WORKDIR so model-loading steps can reuse it
# tokenizer.save_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"))

# max_input_length = 128
# max_target_length = 128

# hf_ds = HFDataset.from_pandas(df, preserve_index=False)

# def tokenize_fn(examples):
#     model_inputs = tokenizer(examples["input_text"], truncation=True, padding="max_length", max_length=max_input_length)
#     labels = tokenizer(examples["target_text"], truncation=True, padding="max_length", max_length=max_target_length)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

# hf_ds = hf_ds.map(tokenize_fn, batched=True, remove_columns=list(df.columns))
# hf_ds.set_format(type="torch")

# # data_collator will be created later with model reference
# print("Tokenized dataset columns:", hf_ds.column_names)
# print("Sample shapes for first item:", {c: hf_ds[0][c].shape for c in hf_ds.column_names if c in hf_ds[0]})


from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

PREPARED_PATH = Path('/content/data/subtaskA_training_prepared.csv')
df = pd.read_csv(PREPARED_PATH)
print("Loaded prepared CSV rows:", len(df))

# <- changed to mt0-xl
MODEL_NAME = "bigscience/mt0-xl"
# some tokenizers for instruction-tuned models work better with use_fast=False
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# --- Add explicit language tokens so "<lang:en>" is a known token ---
# list tokens to add; keep same format you used in inputs ("<lang:en>" etc.)
LANG_TOKENS = ["<lang:en>", "<lang:de>", "<lang:es>", "<lang:ta>", "<lang:kn>"]

# Add tokens to tokenizer (idempotent: only actually adds missing tokens)
added = tokenizer.add_tokens([t for t in LANG_TOKENS if t not in tokenizer.get_vocab()])
print("New tokens added:", added)

# Save tokenizer to WORKDIR so model-loading steps can reuse it
tokenizer.save_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"))

max_input_length = 128
max_target_length = 128

hf_ds = HFDataset.from_pandas(df, preserve_index=False)

def tokenize_fn(examples):
    model_inputs = tokenizer(examples["input_text"], truncation=True, padding="max_length", max_length=max_input_length)
    labels = tokenizer(examples["target_text"], truncation=True, padding="max_length", max_length=max_target_length)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

hf_ds = hf_ds.map(tokenize_fn, batched=True, remove_columns=list(df.columns))
hf_ds.set_format(type="torch")

# data_collator will be created later with model reference
print("Tokenized dataset columns:", hf_ds.column_names)
print("Sample shapes for first item:", {c: hf_ds[0][c].shape for c in hf_ds.column_names if c in hf_ds[0]})


Loaded prepared CSV rows: 7381


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


New tokens added: 5


Map:   0%|          | 0/7381 [00:00<?, ? examples/s]

Tokenized dataset columns: ['input_ids', 'attention_mask', 'labels']
Sample shapes for first item: {'input_ids': torch.Size([128]), 'attention_mask': torch.Size([128]), 'labels': torch.Size([128])}


In [8]:
!pip uninstall -y bitsandbytes triton
!pip install numpy==1.26.4 transformers==4.40.2 peft==0.10.0 accelerate==0.29.3

In [11]:
# # Cell 6
# from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
# from peft import LoraConfig, get_peft_model

# MODEL_NAME = "google/mt5-base"
# USE_LORA = True
# LORA_R = 8
# LORA_ALPHA = 32
# LORA_DROPOUT = 0.1

# print("Loading model (this may take time)...")
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)

# if USE_LORA:
#     lora_config = LoraConfig(
#         r=LORA_R,
#         lora_alpha=LORA_ALPHA,
#         target_modules=["q", "v"],
#         lora_dropout=LORA_DROPOUT,
#         bias="none"
#     )
#     model = get_peft_model(model, lora_config)
#     print("LoRA enabled on model.")

# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# training_args = Seq2SeqTrainingArguments(
#     output_dir=str(OUTPUT_DIR),
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=1,
#     num_train_epochs=3,
#     learning_rate=2e-5,
#     warmup_steps=500,
#     logging_steps=50,
#     save_strategy="steps",
#     save_steps=500,
#     save_total_limit=3,
#     eval_strategy="no",
#     fp16=False,
#     bf16=False,
#     predict_with_generate=False,
#     report_to="none",
#     push_to_hub=False,
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=hf_ds,
#     data_collator=data_collator,
# )

# print("Trainer ready. To train call: trainer.train()")

# Cell 6 (REPLACE)
# from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
# from peft import LoraConfig, get_peft_model

# MODEL_NAME = "google/mt5-base"
# USE_LORA = True
# LORA_R = 8
# LORA_ALPHA = 32
# LORA_DROPOUT = 0.1

# print("Loading model (this may take time)...")
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)

# # Resize token embeddings since we added tokens to tokenizer in Cell 5
# # Make sure we load the same tokenizer that we updated and saved earlier.
# tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"))
# model.resize_token_embeddings(len(tokenizer))
# print("Resized model embeddings to tokenizer length:", len(tokenizer))

# if USE_LORA:
#     lora_config = LoraConfig(
#         r=LORA_R,
#         lora_alpha=LORA_ALPHA,
#         target_modules=["q", "v"],
#         lora_dropout=LORA_DROPOUT,
#         bias="none"
#     )
#     model = get_peft_model(model, lora_config)
#     print("LoRA enabled on model.")

# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# training_args = Seq2SeqTrainingArguments(
#     output_dir=str(OUTPUT_DIR),
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=1,
#     num_train_epochs=3,
#     learning_rate=2e-5,
#     warmup_steps=500,
#     logging_steps=50,
#     save_strategy="steps",
#     save_steps=500,
#     save_total_limit=3,
#     eval_strategy="no",
#     fp16=False,   # you requested FP32 on A100
#     bf16=False,
#     predict_with_generate=False,
#     report_to="none",
#     push_to_hub=False,
# )

# # trainer will be created with hf_ds defined in Cell 5
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=hf_ds,
#     data_collator=data_collator,
# )

# print("Trainer ready. To continue training call trainer.train()")
# from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
# from peft import LoraConfig, get_peft_model

# # mt0-xl model
# MODEL_NAME = "bigscience/mt0-xl"

# USE_LORA = True
# LORA_R = 8
# LORA_ALPHA = 32
# LORA_DROPOUT = 0.1

# print("Loading model (this may take time)...")
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)

# # enable gradient checkpointing (important for large models)
# try:
#     model.gradient_checkpointing_enable()
#     print("Enabled gradient checkpointing on model.")
# except Exception:
#     pass

# # load tokenizer with added language tokens
# tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"), use_fast=False)

# # resize embeddings because we added tokens
# model.resize_token_embeddings(len(tokenizer))
# print("Resized model embeddings to tokenizer length:", len(tokenizer))

# if USE_LORA:
#     lora_config = LoraConfig(
#         r=LORA_R,
#         lora_alpha=LORA_ALPHA,
#         target_modules=["q", "v"],
#         lora_dropout=LORA_DROPOUT,
#         bias="none"
#     )
#     model = get_peft_model(model, lora_config)
#     print("LoRA enabled on model.")

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,
#     padding=True
# )

# training_args = Seq2SeqTrainingArguments(
#     output_dir=str(OUTPUT_DIR),

#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,

#     num_train_epochs=3,
#     learning_rate=2e-5,
#     warmup_steps=500,

#     logging_steps=50,

#     save_strategy="steps",
#     save_steps=500,
#     save_total_limit=3,

#     evaluation_strategy="no",   # ✅ FIXED HERE

#     fp16=True,
#     bf16=False,

#     predict_with_generate=False,
#     report_to="none",
#     push_to_hub=False,
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=hf_ds,
#     data_collator=data_collator,
# )

# print("Trainer ready. To continue training call trainer.train()")


from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer

MODEL_NAME = "bigscience/mt0-xl"

USE_LORA = True
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

print("Loading model (this may take time)...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

# enable gradient checkpointing
try:
    model.gradient_checkpointing_enable()
    print("Enabled gradient checkpointing on model.")
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(
    str(WORKDIR / "tokenizer_with_lang_tokens"),
    use_fast=False
)

model.resize_token_embeddings(len(tokenizer))
print("Resized model embeddings to tokenizer length:", len(tokenizer))

if USE_LORA:
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q","v"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM"
    )

    model = get_peft_model(model, lora_config)
    print("LoRA enabled on model.")


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_steps=500,

    logging_steps=50,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    evaluation_strategy="no",

    fp16=True,
    bf16=False,

    predict_with_generate=False,
    report_to="none",
    push_to_hub=False,
)

# shortt trianing to save gpu resources

# training_args = Seq2SeqTrainingArguments(
#     output_dir=str(OUTPUT_DIR),

#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,

#     max_steps=10,   # 🔴 quick test run

#     learning_rate=2e-5,
#     warmup_steps=10,

#     logging_steps=1,

#     save_strategy="no",

#     evaluation_strategy="no",

#     fp16=True,

#     predict_with_generate=False,
#     report_to="none",
#     push_to_hub=False,
# )

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_ds,
    data_collator=data_collator,
)

print("Trainer ready. To continue training call trainer.train()")

Loading model (this may take time)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Enabled gradient checkpointing on model.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Resized model embeddings to tokenizer length: 250105
LoRA enabled on model.


max_steps is given, it will override any value given in num_train_epochs


Trainer ready. To continue training call trainer.train()


In [12]:
# Cell 7
# import os
# from pathlib import Path

# try:
#     trainer.train()
# except KeyboardInterrupt:
#     print("Training interrupted by user.")
# except Exception as e:
#     print("Training error:", e)
# finally:
#     outdir = OUTPUT_DIR / "final_model"
#     outdir.mkdir(parents=True, exist_ok=True)
#     trainer.save_model(str(outdir))
#     tokenizer.save_pretrained(str(outdir))
#     print("Saved final model + tokenizer to:", outdir)
#     if USE_LORA:
#         # save PEFT adapter weights
#         try:
#             model.save_pretrained(str(OUTPUT_DIR / "lora_adapter"))
#             print("Saved LoRA adapter to:", OUTPUT_DIR / "lora_adapter")
#         except Exception as e:
#             print("Could not save LoRA adapter automatically:", e)

import os
import numpy as np
import torch
from pathlib import Path

# ensure numpy works (prevents "Numpy is not available")
print("NumPy version:", np.__version__)

try:
    trainer.train()

except KeyboardInterrupt:
    print("Training interrupted by user.")

except Exception as e:
    print("Training error:", e)

finally:
    outdir = OUTPUT_DIR / "final_model"
    outdir.mkdir(parents=True, exist_ok=True)

    trainer.save_model(str(outdir))
    tokenizer.save_pretrained(str(outdir))

    print("Saved final model + tokenizer to:", outdir)

    if USE_LORA:
        try:
            model.save_pretrained(str(OUTPUT_DIR / "lora_adapter"))
            print("Saved LoRA adapter to:", OUTPUT_DIR / "lora_adapter")
        except Exception as e:
            print("Could not save LoRA adapter automatically:", e)

NumPy version: 1.26.4


Step,Training Loss
1,27.948700
2,26.959700
3,27.110600
4,27.244500
5,27.185600
6,28.084600
7,27.392900
8,27.918400
9,27.654200
10,26.641700


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:168: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Saved final model + tokenizer to: /content/mt5_lora_output_subtaskA/final_model
Saved LoRA adapter to: /content/mt5_lora_output_subtaskA/lora_adapter


In [15]:
# FREE GPU MEMORY AFTER TRAINING
import torch
import gc

try:
    del trainer
except:
    pass

try:
    del model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")

GPU memory cleared


In [17]:
# Cell 8 (REPLACE)
# import os
# from pathlib import Path
# import torch
# from typing import List, Tuple
# from peft import PeftModel

# # Prefer local saved model if present
# MODEL_LOAD_DIR = OUTPUT_DIR / "final_model"
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # reload tokenizer we saved earlier (with lang tokens)
# tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"))

# # load model if saved, otherwise use the in-memory model if trainer is in scope
# if MODEL_LOAD_DIR.exists() and any(MODEL_LOAD_DIR.iterdir()):
#     print("Loading saved model from", MODEL_LOAD_DIR)
#     # Load base model
#     base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)
#     # Apply LoRA adapter
#     model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR / "lora_adapter"))
# else:
#     print("Using model already in memory (trainer/model).")

# model.to(device)
# model.eval()

# def prepare_instruction_prompt(raw_prompt: str, lang: str = "en") -> str:
#     """
#     Build an instruction-style prompt that tells model exactly what to do:
#       - if prompt contains '___' ask to fill blank with neutral term then answer
#       - otherwise ask to rewrite neutrally and then answer if question-like
#     """
#     # Basic instruction templates — tune as needed
#     if "___" in raw_prompt:
#         instr = "Fill the blank with a gender-neutral term (single or multiword as appropriate) and then answer the request."
#     elif raw_prompt.strip().endswith("?") or "Describe" in raw_prompt or "describe" in raw_prompt:
#         instr = "Rewrite neutrally if needed and then answer the request in one short paragraph."
#     else:
#         instr = "Rewrite neutrally if needed and then provide a short paragraph response."

#     # ensure language tag token is present exactly as in tokenizer
#     lang_token = f"<lang:{lang}>"
#     return f"{lang_token} {instr} Input: {raw_prompt}"

# def extract_filled_word_from_prediction(original_prompt: str, prediction: str) -> Tuple[str,str]:
#     """
#     If original prompt contains '___', try to extract the replacement token(s).
#     Approach (simple heuristic): find the first token(s) in prediction that replace the blank
#     by comparing the sentence boundaries if possible. This is heuristic — may require
#     more robust parsing for complex outputs.
#     Returns (filled_word, possibly_trimmed_prediction).
#     """
#     if "___" not in original_prompt:
#         return None, prediction

#     # naive heuristic: split at sentence punctuation and find the sentence portion matching original
#     try:
#         # build a version of original with placeholder removed to locate context
#         left, right = original_prompt.split("___", 1)
#         # remove punctuation left/right for simpler matching
#         left = left.strip()
#         right = right.strip()
#         # attempt to find the substring in the prediction where left and right anchor exist
#         pred = prediction.strip()
#         # try to find left anchor
#         li = pred.lower().find(left.lower())
#         if li != -1:
#             rest = pred[li + len(left):]
#             # attempt to match right anchor next
#             rj = rest.lower().find(right.lower())
#             if rj != -1:
#                 filled = rest[:rj].strip()
#                 filled = filled.strip(" ,.;:()\"'")
#                 # cleaned full prediction
#                 cleaned_pred = (pred[:li] + left + " " + filled + " " + right + pred[li+len(left)+rj+len(right):]).strip()
#                 return filled, cleaned_pred
#     except Exception:
#         pass

#     # fallback: return None and original prediction
#     return None, prediction

# def generate_texts(prompts: List[str], langs: List[str]=None, max_length: int = 64, num_beams: int = 5) -> List[str]:
#     """
#     Prompts: raw prompt strings (no lang tags).
#     Langs: parallel list of language codes (en/de/es/ta/kn). If None uses 'en'.
#     We will build instruction prompts for each input, tokenize and generate.
#     """
#     if langs is None:
#         langs = ["en"] * len(prompts)
#     assert len(langs) == len(prompts)

#     instruction_prompts = [prepare_instruction_prompt(p, l) for p,l in zip(prompts, langs)]
#     inputs = tokenizer(instruction_prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)

#     with torch.no_grad():
#         out = model.generate(
#             input_ids=inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             max_length=max_length,
#             num_beams=num_beams,
#             early_stopping=True,
#             no_repeat_ngram_size=3
#         )
#     raw_outs = tokenizer.batch_decode(out, skip_special_tokens=True)

#     # If prompts contained blanks, attempt to extract the filled token(s)
#     final_outs = []
#     for orig_prompt, cfg_lang, raw in zip(prompts, langs, raw_outs):
#         if "___" in orig_prompt:
#             filled, cleaned = extract_filled_word_from_prediction(orig_prompt, raw)
#             if filled:
#                 # include both filled token and the full cleaned output in a simple structured form
#                 out_text = f"[filled: {filled}] {cleaned}"
#             else:
#                 out_text = raw
#         else:
#             out_text = raw
#         final_outs.append(out_text)
#     return final_outs

# Cell 8 (REPLACE)
# import os
# from pathlib import Path
# import torch
# from typing import List, Tuple
# from peft import PeftModel

# # Prefer local saved model if present
# MODEL_LOAD_DIR = OUTPUT_DIR / "final_model"
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # reload tokenizer we saved earlier (with lang tokens)
# tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"), use_fast=False)

# # load model if saved, otherwise use the in-memory model if trainer is in scope
# if MODEL_LOAD_DIR.exists() and any(MODEL_LOAD_DIR.iterdir()):
#     print("Loading saved model from", MODEL_LOAD_DIR)
#     # Load base model (ensure same base as training)
#     base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)
#     # Apply LoRA adapter (if saved)
#     model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR / "lora_adapter"))
# else:
#     print("Using model already in memory (trainer/model).")

# model.to(device)
# model.eval()

# def prepare_instruction_prompt(raw_prompt: str, lang: str = "en") -> str:
#     """
#     Build an instruction-style prompt that tells model exactly what to do:
#       - if prompt contains '___' ask to fill blank with neutral term then answer
#       - otherwise ask to rewrite neutrally and then answer if question-like
#     """
#     # Basic instruction templates — tune as needed
#     if "___" in raw_prompt:
#         instr = "Fill the blank with a gender-neutral term (single or multiword as appropriate) and then answer the request."
#     elif raw_prompt.strip().endswith("?") or "Describe" in raw_prompt or "describe" in raw_prompt:
#         instr = "Rewrite neutrally if needed and then answer the request in one short paragraph."
#     else:
#         instr = "Rewrite neutrally if needed and then provide a short paragraph response."

#     # ensure language tag token is present exactly as in tokenizer
#     lang_token = f"<lang:{lang}>"
#     return f"{lang_token} {instr} Input: {raw_prompt}"

# def extract_filled_word_from_prediction(original_prompt: str, prediction: str) -> Tuple[str,str]:
#     """
#     If original prompt contains '___', try to extract the replacement token(s).
#     Approach (simple heuristic): find the first token(s) in prediction that replace the blank
#     by comparing the sentence boundaries if possible. This is heuristic — may require
#     more robust parsing for complex outputs.
#     Returns (filled_word, possibly_trimmed_prediction).
#     """
#     if "___" not in original_prompt:
#         return None, prediction

#     # naive heuristic: split at sentence punctuation and find the sentence portion matching original
#     try:
#         # build a version of original with placeholder removed to locate context
#         left, right = original_prompt.split("___", 1)
#         # remove punctuation left/right for simpler matching
#         left = left.strip()
#         right = right.strip()
#         # attempt to find the substring in the prediction where left and right anchor exist
#         pred = prediction.strip()
#         # try to find left anchor
#         li = pred.lower().find(left.lower())
#         if li != -1:
#             rest = pred[li + len(left):]
#             # attempt to match right anchor next
#             rj = rest.lower().find(right.lower())
#             if rj != -1:
#                 filled = rest[:rj].strip()
#                 filled = filled.strip(" ,.;:()\"'")
#                 # cleaned full prediction
#                 cleaned_pred = (pred[:li] + left + " " + filled + " " + right + pred[li+len(left)+rj+len(right):]).strip()
#                 return filled, cleaned_pred
#     except Exception:
#         pass

#     # fallback: return None and original prediction
#     return None, prediction

# def generate_texts(prompts: List[str], langs: List[str]=None, max_length: int = 64, num_beams: int = 5) -> List[str]:
#     """
#     Prompts: raw prompt strings (no lang tags).
#     Langs: parallel list of language codes (en/de/es/ta/kn). If None uses 'en'.
#     We will build instruction prompts for each input, tokenize and generate.
#     """
#     if langs is None:
#         langs = ["en"] * len(prompts)
#     assert len(langs) == len(prompts)

#     instruction_prompts = [prepare_instruction_prompt(p, l) for p,l in zip(prompts, langs)]
#     inputs = tokenizer(instruction_prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)

#     with torch.no_grad():
#         out = model.generate(
#             input_ids=inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             max_length=max_length,
#             num_beams=num_beams,
#             early_stopping=True,
#             no_repeat_ngram_size=3
#         )
#     raw_outs = tokenizer.batch_decode(out, skip_special_tokens=True)

#     # If prompts contained blanks, attempt to extract the filled token(s)
#     final_outs = []
#     for orig_prompt, cfg_lang, raw in zip(prompts, langs, raw_outs):
#         if "___" in orig_prompt:
#             filled, cleaned = extract_filled_word_from_prediction(orig_prompt, raw)
#             if filled:
#                 # include both filled token and the full cleaned output in a simple structured form
#                 out_text = f"[filled: {filled}] {cleaned}"
#             else:
#                 out_text = raw
#         else:
#             out_text = raw
#         final_outs.append(out_text)
#     return final_outs

# # Cell 8
# import os
# from pathlib import Path
# import torch
# from typing import List, Tuple
# from peft import PeftModel
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# MODEL_NAME = "bigscience/mt0-xl"

# MODEL_LOAD_DIR = OUTPUT_DIR / "final_model"
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # load tokenizer with language tokens
# tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"), use_fast=False)

# if MODEL_LOAD_DIR.exists() and any(MODEL_LOAD_DIR.iterdir()):
#     print("Loading saved model from", MODEL_LOAD_DIR)

#     # load base model
#     base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)

#     # IMPORTANT: resize embeddings BEFORE loading LoRA
#     base_model.resize_token_embeddings(len(tokenizer))

#     # load LoRA adapter
#     model = PeftModel.from_pretrained(
#         base_model,
#         str(OUTPUT_DIR / "lora_adapter")
#     )

# else:
#     print("Using model already in memory.")

# model.to(device)
# model.eval()

# print("Model ready for inference.")

# Cell 8 (fixed loading)

import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_LOAD_DIR = OUTPUT_DIR / "final_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(str(WORKDIR / "tokenizer_with_lang_tokens"), use_fast=False)

print("Loading base model on CPU...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": "cpu"}   # force CPU loading
)

base_model.resize_token_embeddings(len(tokenizer))

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    str(OUTPUT_DIR / "lora_adapter"),
    device_map={"": "cpu"}   # keep on CPU during loading
)

print("Moving model to GPU...")
model = model.to(device)

model.eval()

print("Model ready.")

Loading tokenizer...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading base model on CPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA adapter...
Moving model to GPU...
Model ready.


In [18]:
# # Cell 9
# !pip install -q bert-score sentence-transformers sacrebleu

# from bert_score import score as bertscore_score
# from sentence_transformers import SentenceTransformer, util as st_util
# import sacrebleu
# import numpy as np
# import json
# from collections import defaultdict

# embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# # small language pronoun/lexicon heuristics for GN/GA
# GENDER_LEX = {
#     "en": set(["he","she","his","her","him","woman","man","boy","girl","mother","father","son","daughter","waiter","waitress","actor","actress","king","queen"]),
#     "de": set(["er","sie","sein","ihr","mann","frau","junge","mädchen","mutter","vater","sohn","tochter","kellner","kellnerin","schauspieler","schauspielerin","könig","königin"]),
#     "es": set(["él","ella","su","suyo","mujer","hombre","niño","niña","madre","padre","hijo","hija","camarero","camarera","actor","actriz","rey","reina"]),
#     "ta": set(["அவன்","அவள்","அவர்","ஆண்","பெண்"]),  # minimal Tamil markers
#     "kn": set(["ಅವನು","ಅವಳು","ಅವರು","ಗಂಡ","ಹೆಣ್ಣು"])  # minimal Kannada markers
# }

# PRONOUNS = {
#     "en": set(["he","she","they","him","her","them","his","hers","their","theirs"]),
#     "de": set(["er","sie","sie","ihn","ihr","sein","ihr","ihnen"]),
#     "es": set(["él","ella","ellos","ellas","lo","la","los","las","su","sus"]),
#     "ta": set(["அவன்","அவள்","அவர்கள்"]),
#     "kn": set(["ಅವನು","ಅವಳು","ಅವರು"])
# }

# def compute_bleu(references: List[str], hypotheses: List[str]):
#     refs = [[r] for r in references]  # sacrebleu expects list of refs per hypothesis? we'll use corpus_bleu
#     # sacrebleu.corpus_bleu expects list of hypotheses and list of list-of-refs
#     bleu = sacrebleu.corpus_bleu(hypotheses, [references])
#     return {"bleu": bleu.score}

# def compute_chrf(references: List[str], hypotheses: List[str]):
#     chrf = sacrebleu.corpus_chrf(hypotheses, [references])
#     return {"chrf": chrf.score}

# def compute_bertscore(references: List[str], hypotheses: List[str], lang="en"):
#     P, R, F1 = bertscore_score(hypotheses, references, lang="en", return_hash=False)
#     return {"bertscore_f1": float(F1.mean().cpu().numpy())}

# def compute_embedding_cosine(references: List[str], hypotheses: List[str]):
#     ref_emb = embed_model.encode(references, convert_to_tensor=True)
#     hyp_emb = embed_model.encode(hypotheses, convert_to_tensor=True)
#     sims = st_util.cos_sim(ref_emb, hyp_emb).diagonal().cpu().numpy()
#     return {"embed_cosine_mean": float(np.mean(sims)), "embed_cosine_std": float(np.std(sims))}

# def gender_neutrality_score(hypotheses: List[str], lang="en"):
#     lex = GENDER_LEX.get(lang, set())
#     scores = []
#     for h in hypotheses:
#         tokens = [t.lower().strip(".,:;!?\"'") for t in h.split()]
#         found = any(tok in lex for tok in tokens)
#         scores.append(0 if found else 1)
#     return {"gn_mean": float(np.mean(scores)), "gn_scores": scores}

# def gender_assumption_score(source_texts: List[str], hypotheses: List[str], lang="en"):
#     pron = PRONOUNS.get(lang, set())
#     scores = []
#     for s,h in zip(source_texts, hypotheses):
#         s_toks = set([t.lower().strip(".,:;!?\"'") for t in s.split()])
#         h_toks = set([t.lower().strip(".,:;!?\"'") for t in h.split()])
#         # if source already contains gender tokens, we don't penalize
#         if any(t in pron for t in s_toks):
#             scores.append(1.0)
#             continue
#         # if hypothesis introduces a gendered pronoun -> bad (score 0)
#         if any(t in pron for t in h_toks):
#             scores.append(0.0)
#         else:
#             scores.append(1.0)
#     return {"ga_mean": float(np.mean(scores)), "ga_scores": scores}

# Cell 9
import re
import torch
from typing import List, Tuple

# Safety: ensure tokenizer has lang tokens; if not, add and resize
LANG_TOKENS = ["<lang:en>", "<lang:de>", "<lang:es>", "<lang:ta>", "<lang:kn>"]
_added = [t for t in LANG_TOKENS if t not in tokenizer.get_vocab()]
if _added:
    tokenizer.add_tokens(_added)
    try:
        model.resize_token_embeddings(len(tokenizer))
        print("Added lang tokens and resized model embeddings:", _added)
    except Exception as e:
        print("Warning: unable to resize embeddings (if you plan to retrain, resize before training).", e)

model.to(device)
model.eval()

# Few-shot examples (short, instruction -> output) — optional, helpful for instruction following.
# Keep these short and language-aware; here only English examples. You can expand for other languages later.
FEWSHOT_EN = [
    # (instruction, example_input, example_output)
    ("Rewrite neutrally and preserve meaning.",
     "The mailman delivered the packages. Rewrite to be gender-neutral.",
     "The mail carrier delivered the packages."),
    ("Fill the blank with a gender-neutral term and then answer.",
     "The ___ is a skilled nurse working in a busy hospital. Describe their daily tasks.",
     "The person is a skilled nurse working in a busy hospital. They coordinate patient care, administer medications, and liaise with physicians.")
]

# decide task type heuristics
def decide_task(prompt: str) -> str:
    p = prompt.lower()
    # rewrite keywords
    rewrite_keywords = ["rewrite", "make .*inclusive", "make inclusive", "neutral", "make gender-neutral", "make gender neutral", "rewrit", "inclusive"]
    # generate keywords
    generate_keywords = ["describe", "explain", "write a", "write an", "write", "bio", "job ad", "advert", "describe their", "describe the", "tell me about", "what is", "list"]
    # blank detection
    if "___" in p or "_ _ _" in p:
        return "fill_and_generate"
    for kw in rewrite_keywords:
        if re.search(kw, p):
            return "rewrite"
    for kw in generate_keywords:
        if kw in p:
            return "generate"
    # default: generate (safer)
    return "generate"

# build instruction prompt
def build_instruction_prompt(raw_prompt: str, lang: str = "en", few_shot: bool = False) -> str:
    task = decide_task(raw_prompt)
    lang_token = f"<lang:{lang}>"
    if task == "rewrite":
        instr = ("Rewrite the following text in the same language to be gender-neutral and "
                 "preserve the original meaning. Output only the rewritten text, no explanations.")
    elif task == "fill_and_generate":
        instr = ("Fill the blank with a gender-neutral term (single or multiword as appropriate), "
                 "then answer the rest of the prompt in one concise paragraph. Output the filled word "
                 "and the paragraph; do not include meta text.")
    else:  # generate
        instr = ("Produce a concise gender-neutral paragraph answering the prompt. "
                 "Keep it coherent, factual, and in the same language. Do not add meta commentary.")
    # few-shot prefix if requested and language is english (can expand later)
    fs = ""
    if few_shot and lang == "en":
        for t, ex_in, ex_out in FEWSHOT_EN:
            fs += f"Example: {t}\nInput: {ex_in}\nOutput: {ex_out}\n\n"
    # final composed prompt
    composed = f"{lang_token} {fs}{instr} Input: {raw_prompt}"
    return composed

# clean model output: remove sentinel tokens and repeats, strip "Input:" echoes
SENTINEL_RE = re.compile(r"<extra_id_\d+>")
def clean_output(text: str) -> str:
    # remove sentinels
    t = SENTINEL_RE.sub("", text)
    # remove repeated "Input:" if model echoed it
    t = re.sub(r"Input:\s*", "", t, flags=re.IGNORECASE)
    # remove stray multiple whitespace / leading punctuation
    t = t.strip()
    t = re.sub(r"\s{2,}", " ", t)
    # remove leftover "Output:" labels
    t = re.sub(r"^output:\s*", "", t, flags=re.IGNORECASE)
    # sometimes model returns just the lang tag — strip that
    t = re.sub(r"^<lang:[a-z]{2}>\s*", "", t)
    return t.strip()

# helper to extract filled token if blank exists
def extract_filled(original_prompt: str, generated_text: str) -> Tuple[str, str]:
    if "___" not in original_prompt:
        return None, generated_text
    # try to find replacement between left and right anchors
    try:
        left, right = original_prompt.split("___", 1)
        left = left.strip()
        right = right.strip()
        pred = generated_text.strip()
        # locate left in prediction
        li = pred.lower().find(left.lower())
        if li != -1:
            rest = pred[li + len(left):]
            rj = rest.lower().find(right.lower()) if right else -1
            if rj != -1:
                filled = rest[:rj].strip()
                # clean filled
                filled = re.sub(r"[^\w\s\-']", "", filled).strip()
                cleaned_pred = pred  # already pred contains filled
                return filled, cleaned_pred
    except Exception:
        pass
    return None, generated_text

# generate with fallbacks
def generate_instruction_responses(prompts: List[str], langs: List[str]=None,
                                   max_length: int = 160, num_beams: int = 5,
                                   few_shot: bool = True) -> List[str]:
    if langs is None:
        langs = ["en"] * len(prompts)
    assert len(langs) == len(prompts)
    composed_prompts = [build_instruction_prompt(p, l, few_shot=few_shot) for p,l in zip(prompts, langs)]
    inputs = tokenizer(composed_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0
        )
    raw = tokenizer.batch_decode(out, skip_special_tokens=False)
    cleaned = [clean_output(r) for r in raw]

    # fallback: if cleaned output too short or only empty, regenerate with stronger settings
    results = []
    for orig, prmpt, lang, r in zip(prompts, composed_prompts, langs, cleaned):
        if len(r.strip()) < 5:
            # regenerate with larger max and beams and force skip special tokens
            inputs2 = tokenizer(prmpt + " Please respond only with the required answer.", return_tensors="pt",
                                truncation=True, padding=True, max_length=512).to(device)
            with torch.no_grad():
                out2 = model.generate(
                    input_ids=inputs2["input_ids"],
                    attention_mask=inputs2["attention_mask"],
                    max_length=max(256, max_length),
                    num_beams=8,
                    early_stopping=True,
                    no_repeat_ngram_size=2,
                    length_penalty=1.0
                )
            r2 = tokenizer.batch_decode(out2, skip_special_tokens=False)[0]
            r2 = clean_output(r2)
            r = r2 if len(r2.strip()) > len(r.strip()) else r

        # extract filled token if prompt had '___'
        filled, final_text = extract_filled(orig, r)
        if filled:
            results.append(f"[filled: {filled}] {final_text}")
        else:
            results.append(r)
    return results


In [19]:
# Clear training memory before inference
import torch
import gc

try:
    del trainer
except:
    pass

try:
    del model
except:
    pass

torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared.")

GPU memory cleared.


In [21]:
# Reload model for inference

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

MODEL_NAME = "bigscience/mt0-xl"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    str(WORKDIR / "tokenizer_with_lang_tokens"),
    use_fast=False
)

print("Loading base model on CPU...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": "cpu"}  # force CPU load
)

base_model.resize_token_embeddings(len(tokenizer))

print("Loading LoRA adapter (CPU)...")
model = PeftModel.from_pretrained(
    base_model,
    str(OUTPUT_DIR / "lora_adapter"),
    device_map={"": "cpu"}
)

print("Moving model to GPU...")
model = model.to(device)
model.eval()

print("Model ready for inference.")

Loading tokenizer...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading base model on CPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA adapter (CPU)...
Moving model to GPU...
Model ready for inference.


In [24]:
# NEW CELL A — Fix tokenizer / decoder configuration

print("Checking tokenizer configuration...")

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)

# ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print("Pad token was missing — set to EOS token.")

# set decoder start token
model.config.decoder_start_token_id = tokenizer.pad_token_id

print("decoder_start_token_id set to:", model.config.decoder_start_token_id)

Checking tokenizer configuration...
pad_token: <pad>
pad_token_id: 0
eos_token: </s>
eos_token_id: 1
decoder_start_token_id set to: 0


In [25]:
# NEW CELL B — Safer generation function

import torch

def generate_instruction_responses(prompts, langs=None, max_length=120):

    if langs is None:
        langs = ["en"] * len(prompts)

    outputs = []

    for prompt, lang in zip(prompts, langs):

        model_input = f"<lang:{lang}> {prompt}"

        inputs = tokenizer(
            model_input,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(device)

        with torch.no_grad():

            generated = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],

                max_length=max_length,

                # safer generation settings
                num_beams=1,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,

                early_stopping=False,
                no_repeat_ngram_size=3,

                decoder_start_token_id=model.config.decoder_start_token_id
            )

        text = tokenizer.decode(
            generated[0],
            skip_special_tokens=True
        ).strip()

        outputs.append(text)

    return outputs

print("Generation function replaced.")

Generation function replaced.


In [27]:
# # Cell 10
# import pandas as pd
# import json
# from collections import defaultdict

# TEST_DIR = WORKDIR / "tests"
# TEST_DIR.mkdir(exist_ok=True)
# # expects CSVs in TEST_DIR or single CSV path. Each CSV should have at least columns: Test Case ID, Input Prompt
# # optional column: Reference  (if present, will compute automatic metrics)

# def run_evaluation_on_csv(test_csv_path: Path, lang_hint=None, max_gen_len=64, num_beams=5, save_outputs=True):
#     df = pd.read_csv(test_csv_path)
#     # ensure columns
#     if "Input Prompt" not in df.columns and "Input" not in df.columns and "Prompt" not in df.columns:
#         raise ValueError("Test CSV must have column named 'Input Prompt' (or 'Input'/'Prompt')")
#     # unify column name
#     if "Input Prompt" in df.columns:
#         prompts = df["Input Prompt"].astype(str).tolist()
#     elif "Input" in df.columns:
#         prompts = df["Input"].astype(str).tolist()
#     else:
#         prompts = df["Prompt"].astype(str).tolist()

#     # build model inputs: if blank marker '___' present, keep prompt as-is because model was trained with <lang:xx> + full sentence style
#     if "lang" in df.columns:
#         langs = df["lang"].fillna(lang_hint).astype(str).tolist()
#     else:
#         langs = [lang_hint if lang_hint is not None else "en"] * len(prompts)

#     model_inputs = [f"<lang:{l}> {p}" for l,p in zip(langs, prompts)]
#     # generate
#     outputs = []
#     batch_size = 16
#     for i in range(0, len(model_inputs), batch_size):
#         batch = model_inputs[i:i+batch_size]
#         outs = generate_texts(batch, max_length=max_gen_len, num_beams=num_beams)
#         outputs.extend(outs)

#     df["prediction"] = outputs

#     results = {"file": str(test_csv_path), "n": len(df)}

#     if "Reference" in df.columns or "Reference Text" in df.columns or "Target" in df.columns:
#         ref_col = "Reference" if "Reference" in df.columns else ("Reference Text" if "Reference Text" in df.columns else "Target")
#         references = df[ref_col].astype(str).tolist()
#         hypotheses = df["prediction"].astype(str).tolist()
#         results.update(compute_bleu(references, hypotheses))
#         results.update(compute_chrf(references, hypotheses))
#         try:
#             results.update(compute_bertscore(references, hypotheses))
#         except Exception as e:
#             print("BERTScore failed:", e)
#         try:
#             results.update(compute_embedding_cosine(references, hypotheses))
#         except Exception as e:
#             print("Embedding sim failed:", e)
#         # GN/GA per-language
#         # compute per-language means
#         lang_groups = defaultdict(list)
#         for idx,row in df.iterrows():
#             lang_groups[row.get("lang","en")].append(idx)
#         gn_per_lang = {}
#         ga_per_lang = {}
#         for l, idxs in lang_groups.items():
#             hyps = [df.loc[i,"prediction"] for i in idxs]
#             srcs = [df.loc[i,"Input Prompt"] if "Input Prompt" in df.columns else df.loc[i,"Input"] for i in idxs]
#             gn = gender_neutrality_score(hyps, lang=l)
#             ga = gender_assumption_score(srcs, hyps, lang=l)
#             gn_per_lang[l] = gn["gn_mean"]
#             ga_per_lang[l] = ga["ga_mean"]
#         results["gn_per_lang"] = gn_per_lang
#         results["ga_per_lang"] = ga_per_lang
#     else:
#         # no references: compute GN/GA only
#         hyps = df["prediction"].astype(str).tolist()
#         gn = gender_neutrality_score(hyps, lang="en")
#         results.update(gn)

#     if save_outputs:
#         out_path = WORKDIR / f"predictions_{Path(test_csv_path).stem}.csv"
#         df.to_csv(out_path, index=False)
#         stats_path = WORKDIR / f"metrics_{Path(test_csv_path).stem}.json"
#         with open(stats_path, "w") as fh:
#             json.dump(results, fh, indent=2)
#         print("Saved predictions to", out_path)
#         print("Saved metrics to", stats_path)

#     return df, results

# # Example usage for a single test CSV if you have one:
# # sample_test_csv = TEST_DIR / "test_german_fill.csv"
# # df_out, metrics = run_evaluation_on_csv(sample_test_csv, lang_hint="de")

# Cell 10
# Example test cases you provided; replace or load CSVs as needed
# tests = [
#     ("The nurse enters the hospital ward. Describe their routine and responsibilities.", "en"),
#     ("A teacher starts the first day of class. Describe their interaction with students.", "en"),
#     ("The ___ is a skilled nurse working in a busy hospital. Describe their daily tasks.", "en"),
#     ("___ completed a degree in computer science and joined a startup. Describe their role.", "en"),
#     ("Die Pflegefachkraft betritt die Krankenhausstation. Beschreiben Sie den Arbeitsalltag und die Aufgaben.", "de"),
#     ("Escribe una frase: ___ (solo la palabra) para 'chico recogepelotas' y luego úsala en una oración.", "es"),  # example spanish
# ]

# prompts = [t[0] for t in tests]
# langs = [t[1] for t in tests]

# print("Generating outputs...")
# outs = generate_instruction_responses(prompts, langs, max_length=200, num_beams=6, few_shot=True)
# for i,(p,l,o) in enumerate(zip(prompts, langs, outs)):
#     print(f"\nTest {i+1} | lang={l}\nPrompt: {p}\nPrediction: {o}\n")

# Cell 10

tests = [
    ("The nurse enters the hospital ward. Describe their routine and responsibilities.", "en"),
    ("A teacher starts the first day of class. Describe their interaction with students.", "en"),
    ("The ___ is a skilled nurse working in a busy hospital. Describe their daily tasks.", "en"),
    ("___ completed a degree in computer science and joined a startup. Describe their role.", "en"),
    ("Die Pflegefachkraft betritt die Krankenhausstation. Beschreiben Sie den Arbeitsalltag und die Aufgaben.", "de"),
    ("Escribe una frase: ___ (solo la palabra) para 'chico recogepelotas' y luego úsala en una oración.", "es"),
]

print("Generating outputs...")

outputs = []

for i,(prompt,lang) in enumerate(tests):

    torch.cuda.empty_cache()

    out = generate_instruction_responses(
        [prompt],
        [lang],
        max_length=120,
        # num_beams=1,
        # few_shot=True
    )[0]

    outputs.append(out)

    print(f"\nTest {i+1} | lang={lang}")
    print("Prompt:", prompt)
    print("Prediction:", out)


Generating outputs...

Test 1 | lang=en
Prompt: The nurse enters the hospital ward. Describe their routine and responsibilities.
Prediction: nurses enter the hospital ward with their nurse

Test 2 | lang=en
Prompt: A teacher starts the first day of class. Describe their interaction with students.
Prediction: They start the class by teaching their students.

Test 3 | lang=en
Prompt: The ___ is a skilled nurse working in a busy hospital. Describe their daily tasks.
Prediction: They do the following: Remove a blanket, wash the stomach, and wash the mouth. Take care of a patient when they are ill. Prepare the nurse's supplies.

Test 4 | lang=en
Prompt: ___ completed a degree in computer science and joined a startup. Describe their role.
Prediction: a computer programmer

Test 5 | lang=de
Prompt: Die Pflegefachkraft betritt die Krankenhausstation. Beschreiben Sie den Arbeitsalltag und die Aufgaben.
Prediction: . Die nurse's routines

Test 6 | lang=es
Prompt: Escribe una frase: ___ (solo la 

In [14]:
TEST_DIR = Path("/content/drive/MyDrive/Gender Neutral/data/raw/Test")

In [17]:
# Cell 11
from pathlib import Path
test_files = sorted([p for p in TEST_DIR.glob("*.csv")])
if not test_files:
    print("No test CSVs found in", TEST_DIR)
else:
    all_metrics = {}
    for p in test_files:
        print("Running:", p)
        try:
            df_out, metrics = run_evaluation_on_csv(p, lang_hint=None, max_gen_len=80, num_beams=5, save_outputs=True)
            all_metrics[p.name] = metrics
        except Exception as e:
            print("Failed on", p, e)
    # aggregate simple summary
    summary_path = WORKDIR / "all_test_metrics_summary.json"
    with open(summary_path, "w") as fh:
        json.dump(all_metrics, fh, indent=2)
    print("Wrote summary to", summary_path)
    for k,v in all_metrics.items():
        print(k, "->", {kk:vv for kk,vv in v.items() if kk in ("bleu","chrf","bertscore_f1","embed_cosine_mean","gn_per_lang","ga_per_lang")})


No test CSVs found in /content/test


In [18]:
# Edit: replace load_metric usage
!pip install evaluate
from datasets import Dataset, DatasetDict
import evaluate

# Cell 12 - install & imports
!pip install -q "transformers[torch,sentencepiece]>=4.30.0" datasets evaluate accelerate sentencepiece sacrebleu bert-score

import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import matplotlib.pyplot as plt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00


### newly added

In [16]:
# Cell 13 - config
base_path = '/content/drive/My Drive/Gender Neutral/data/raw/'  # your path
model_name_or_path = "google/mt5-base"   # recommended: multilingual mT5 base
output_dir = "/content/drive/My Drive/Gender Neutral/outputs/mt5-base-taskA"
os.makedirs(output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [ ]:
# Cell 14 - load CSVs (adjust to your filenames/column names)
# This will scan csv files in base_path and subfolders and concatenate.
def load_all_csvs(path):
    rows = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.lower().endswith(".csv"):
                fp = os.path.join(root, f)
                df = pd.read_csv(fp)
                print("Loaded:", fp, "shape:", df.shape)
                # Attempt to map columns: try commonly used names
                if 'Counterfactual Sentence' in df.columns and 'Biased Sentence' in df.columns:
                    # convert to input/target
                    df = df.rename(columns={'Biased Sentence':'input_text', 'Counterfactual Sentence':'target_text'})
                elif 'non-inclusive German' in df.columns and 'inclusive German' in df.columns:
                    df = df.rename(columns={'non-inclusive German':'input_text', 'inclusive German':'target_text'})
                # If already has input_text/target_text use directly
                if 'input_text' in df.columns and 'target_text' in df.columns:
                    rows.append(df[['input_text','target_text']])
                else:
                    # fallback: try first two text columns
                    text_cols = [c for c in df.columns if df[c].dtype == object]
                    if len(text_cols) >= 2:
                        df = df.rename(columns={text_cols[0]:'input_text', text_cols[1]:'target_text'})
                        rows.append(df[['input_text','target_text']])
                    else:
                        print("Skipping file (no suitable text columns):", fp)
    if not rows:
        raise ValueError("No suitable CSVs found or recognized columns.")
    full = pd.concat(rows, ignore_index=True)
    full = full.dropna(subset=['input_text','target_text']).reset_index(drop=True)
    return full

df_all = load_all_csvs(base_path)
print("Total examples:", len(df_all))
# quick sample
print(df_all.sample(3))


Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Counterfactual Sentence Pairs.xlsx - CounterFactual Data.csv shape: (726, 4)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/data.csv shape: (250, 4)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Gender Neutral Pairs/Gender Neutral Pairs.xlsx - Sheet1.csv shape: (693, 3)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Gender Neutral Pairs/Gender_Neutral_Pairs_Spanish.xlsx - Sheet1.csv shape: (200, 5)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Gender Neutral Pairs/Gender-Neutral Pairs Tamil.xlsx - இதில் உள்ள ஆங்கிலத்திற்கு இணையா.csv shape: (742, 5)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Gender Neutral Pairs/Gender_Neutral_Pairs_Kannada.xlsx - Sheet1.csv shape: (693, 5)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Inclusive/Gender Neutral Sentence Pairs.xlsx - Inclusive Pairs.csv shape: (1073, 3)
Loaded: /content/drive/My Drive/Gender Neutral/data/raw/Inclusive/Inclu

In [ ]:
# Cell 15 - splits
from sklearn.model_selection import train_test_split
train_df, temp = train_test_split(df_all, test_size=0.20, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42, shuffle=True)

hf_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})
print(hf_datasets)


DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 6177
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 772
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 773
    })
})


In [ ]:
!pip install -q peft

In [ ]:
# Cell 16 - tokenizer & preprocess
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)

max_input_length = 256
max_target_length = 128
prefix = "paraphrase: "  # you can change to "Rewrite to gender-neutral: " if you prefer

def preprocess_function(examples):
    inputs = [prefix + ex for ex in examples["input_text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = hf_datasets.map(preprocess_function, batched=True, remove_columns=hf_datasets["train"].column_names)
tokenized = tokenized.shuffle(seed=42)
print(tokenized)


Map:   0%|          | 0/6177 [00:00<?, ? examples/s]

Map:   0%|          | 0/772 [00:00<?, ? examples/s]

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6177
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 772
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 773
    })
})


In [ ]:
# Cell 17 - model + LoRA + data collator

!pip install -q peft

from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float32
)

model.config.tie_word_embeddings = False

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q", "v"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model = model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt"
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 1,769,472 || all params: 968,342,784 || trainable%: 0.1827


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    num_train_epochs=3,

    fp16=False,
    predict_with_generate=False,   # 🔑 no generation during training
    include_for_metrics=[],        # 🔑 no logits accumulation
    load_best_model_at_end=False,

    save_total_limit=3,
    push_to_hub=False
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=None   # 🔑 disable metrics during training
)


In [ ]:
# Cell 19 - train (SAFE with LoRA)
train_result = trainer.train()
trainer.save_model(output_dir)
trainer.save_state()


Step,Training Loss,Validation Loss
500,39.453223,12.297471
1000,20.791882,9.480881


In [ ]:
# FINAL TEST GENERATION (MEMORY SAFE)

import torch
from tqdm import tqdm
import numpy as np

model.eval()

test_loader = torch.utils.data.DataLoader(
    tokenized["test"],
    batch_size=4,        # 🔑 keep small
    shuffle=False,
    collate_fn=data_collator
)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        generated = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=128,
            num_beams=4
        )

        # Decode predictions and labels separately to handle variable lengths
        decoded_preds = tokenizer.batch_decode(generated.cpu().numpy(), skip_special_tokens=True)
        # Labels usually have -100 for padding, replace before decoding
        labels_np = batch["labels"].cpu().numpy()
        labels_np = np.where(labels_np != -100, labels_np, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels_np, skip_special_tokens=True)

        all_preds.extend(decoded_preds)
        all_labels.extend(decoded_labels)

# Compute metrics using the decoded texts. This assumes `compute_metrics` is defined elsewhere.
# If `compute_metrics` expects token IDs, then padding and clipping are necessary before calling it.
# Assuming a simple text-based metric calculation is intended here:
# Note: `compute_metrics` function is not defined in the current notebook state. It needs to be defined.
# For now, I'll print the decoded predictions and labels.
print("\nSample decoded predictions:", all_preds[:5])
print("Sample decoded labels:", all_labels[:5])

# Placeholder for metrics calculation (needs a defined `compute_metrics` function)
# metrics = compute_metrics((all_preds, all_labels))
# print(metrics)


100%|██████████| 194/194 [00:45<00:00,  4.26it/s]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (773,) + inhomogeneous part.

In [ ]:
# Cell 20 - evaluation and test predictions
val_metrics = trainer.evaluate(tokenized["validation"], max_length=max_target_length, num_beams=4)
print("Validation metrics:", val_metrics)
trainer.save_metrics("validation", val_metrics)

test_metrics = trainer.evaluate(tokenized["test"], max_length=max_target_length, num_beams=4)
print("Test metrics:", test_metrics)
trainer.save_metrics("test", test_metrics)

# Generate predictions for test set and save csv
preds = trainer.predict(tokenized["test"], max_length=max_target_length, num_beams=4)
pred_ids = preds.predictions
decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
# original inputs and references:
test_df_out = pd.DataFrame({
    "input": hf_datasets["test"]["input_text"],
    "reference": hf_datasets["test"]["target_text"],
    "prediction": decoded_preds
})
test_csv_path = os.path.join(output_dir, "test_predictions.csv")
test_df_out.to_csv(test_csv_path, index=False)
print("Saved test predictions to:", test_csv_path)


In [ ]:
# Cell 21 - simple plots
history = trainer.state.log_history
# extract loss and eval metrics over time
loss_steps = [h for h in history if "loss" in h]
eval_steps = [h for h in history if "eval_loss" in h or "eval_rougeL" in h]

# plot loss
plt.figure(figsize=(8,4))
steps = [h.get("step") for h in loss_steps if h.get("loss") is not None]
losses = [h.get("loss") for h in loss_steps if h.get("loss") is not None]
plt.plot(steps, losses, label="train_loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training Loss")
plt.grid(True)
plt.savefig(os.path.join(output_dir, "train_loss.png"))
plt.show()

# plot rougeL over eval steps (if available)
rouge_steps = [h for h in history if "eval_rougeL" in h]
if rouge_steps:
    plt.figure(figsize=(8,4))
    steps = [h["step"] for h in rouge_steps]
    rvals = [h["eval_rougeL"] for h in rouge_steps]
    plt.plot(steps, rvals, label="eval_rougeL")
    plt.xlabel("step")
    plt.ylabel("rougeL")
    plt.title("Validation ROUGE-L over time")
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "eval_rougeL.png"))
    plt.show()



# Saving


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/LT_EDI_2026_SubtaskA"
os.makedirs(BASE_DIR, exist_ok=True)

MODEL_DIR = os.path.join(BASE_DIR, "model")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")
LOGS_DIR = os.path.join(BASE_DIR, "logs")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")

for d in [MODEL_DIR, METRICS_DIR, LOGS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Project directories ready")


✅ Project directories ready


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/LT_EDI_2026_SubtaskA"
os.makedirs(BASE_DIR, exist_ok=True)

MODEL_DIR = os.path.join(BASE_DIR, "model")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")
LOGS_DIR = os.path.join(BASE_DIR, "logs")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")

for d in [MODEL_DIR, METRICS_DIR, LOGS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Project directories ready")


✅ Project directories ready


In [ ]:
trainer.model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("✅ Model + tokenizer saved to Drive")


✅ Model + tokenizer saved to Drive


In [ ]:
model.save_pretrained(os.path.join(MODEL_DIR, "lora_adapter"))


In [ ]:
import pandas as pd

loss_df = pd.DataFrame(trainer.state.log_history)
loss_path = os.path.join(LOGS_DIR, "training_logs.csv")
loss_df.to_csv(loss_path, index=False)

print("✅ Training logs saved")


✅ Training logs saved


In [ ]:
import json
import math
import numpy as np

# Placeholder values as these were not computed in the notebook for this section
# You would typically get these from the trainer.evaluate() call or a custom evaluation loop
eval_loss = trainer.state.log_history[-1].get('eval_loss', 0.0) # Using last recorded eval_loss if available
sim_scores = [0.0] # Placeholder, needs actual calculation
gifi_score = 0.0 # Placeholder, needs actual calculation
accuracy = 0.0 # Placeholder, needs actual calculation

metrics = {
    "eval_loss": float(eval_loss),
    "perplexity": float(math.exp(eval_loss)) if eval_loss > 0 else float('inf'), # Avoid log(0) or negative eval_loss
    "mean_semantic_similarity": float(np.mean(sim_scores)) if sim_scores else 0.0,
    "gifi_score": float(gifi_score),
    "replacement_accuracy": float(accuracy)
}

metrics_path = os.path.join(METRICS_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("✅ Evaluation metrics saved")

NameError: name 'eval_loss' is not defined

In [ ]:
import shutil

zip_path = "/content/drive/MyDrive/LT_EDI_2026_SubtaskA_BACKUP.zip"
shutil.make_archive(zip_path.replace(".zip",""), 'zip', BASE_DIR)

print("✅ Full backup ZIP created")


✅ Full backup ZIP created


# LOADING MODEL BACK FROM SAVED CHECKPOINT

In [ ]:
!pip install -q transformers accelerate peft sentence-transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import json
import numpy as np
import math

from transformers import AutoTokenizer

MODEL_DIR = "/content/drive/MyDrive/LT_EDI_2026_SubtaskA/model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print("✅ Tokenizer loaded")


Mounted at /content/drive
✅ Tokenizer loaded


In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

BASE_MODEL = "google/mt5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    model,
    f"{MODEL_DIR}/lora_adapter"
)

model.eval()
print("✅ Base model + LoRA adapter loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Base model + LoRA adapter loaded
